# Tool catalog: notebook outputs and canvas

Run these cells in order in a disposable JupyterLab project with its live kernel and selected ChatGPT connection. The PNG, SVG, text, and structured outputs below are generated here and are actual live notebook outputs. Select some text in this editor before `read_selection` if you want a nonempty result. Canvas controls depend on whether JupyterLab keeps the plain HTML canvas in its output; an unsupported result is honest and does not justify substituting a fake canvas. Wait for each browser receipt before inspecting it. The final AI questions request real tool calls against these exact output cells.


In [ ]:
from io import BytesIO
from pathlib import Path
import hashlib
from PIL import Image as PILImage
from IPython.display import display, Image, SVG, HTML, Javascript
from nbinlineai.tools import (read_notebook_view, read_selection, list_outputs, read_output, export_output, list_canvases, capture_canvas, export_canvas, start_canvas, capture_notebook_region, operation_status, release_media, stop_source)
image = PILImage.new('RGB', (8, 8), '#d4402f')
stream = BytesIO(); image.save(stream, format='PNG')
PNG_BYTES = stream.getvalue()
SVG_MARKUP = '<svg xmlns="http://www.w3.org/2000/svg" width="24" height="18"><rect width="24" height="18" fill="#2542a3"/></svg>'
print('Disposable PNG and original SVG are ready; no file was saved.')

In [ ]:
display(Image(data=PNG_BYTES))

In [ ]:
print('Existing output text: red square and blue vector')

In [ ]:
display(SVG(data=SVG_MARKUP))

### Existing structured output

This small data-resource MIME output is read from the live model; exporting it returns exact UTF-8 JSON bytes and may save a new `.json` file. No renderer or code execution is triggered by discovery.


In [ ]:
DATA_RESOURCE = {'schema': {'fields': [{'name': 'color', 'type': 'string'}]}, 'data': [{'color': 'é😀'}]}
display({'application/vnd.dataresource+json': DATA_RESOURCE, 'text/plain': 'One color row'}, raw=True)


## Inspect the existing notebook

The first five tools work with live model state. They do not run an output cell for you. The IDs below belong to this copied notebook.

In [ ]:
view = read_notebook_view()

In [ ]:
print('Notebook view:', view.status, view.result, view.error)


In [ ]:
selection = read_selection(max_chars=120)

In [ ]:
print('Selection:', selection.status, selection.result, selection.error)
print('An empty selection is expected if no text was selected in this editor.')


In [ ]:
raster_listing = list_outputs('output-raster')
text_listing = list_outputs('output-text')
vector_listing = list_outputs('output-vector')

In [ ]:
for label, receipt in [('raster', raster_listing), ('text', text_listing), ('vector', vector_listing)]: print(label, receipt.status, receipt.result, receipt.error)
raster_ref = raster_listing.result['outputs'][0] if raster_listing.result else None
text_ref = text_listing.result['outputs'][0] if text_listing.result else None
vector_ref = vector_listing.result['outputs'][0] if vector_listing.result else None

In [ ]:
text_read = read_output(text_ref['cell_id'], text_ref['output_id'], text_ref['revision'], max_chars=120) if text_ref else None

In [ ]:
print(text_read.status, text_read.result, text_read.error) if text_read else print('List the text output first.')

In [ ]:
raster_export = export_output(raster_ref['cell_id'], raster_ref['output_id'], raster_ref['revision']) if raster_ref else None
vector_export = export_output(vector_ref['cell_id'], vector_ref['output_id'], vector_ref['revision'], mime='image/svg+xml') if vector_ref else None

In [ ]:
print('Raster:', raster_export.status, type(raster_export.result).__name__, getattr(raster_export.result, 'size', None), raster_export.error) if raster_export else print('List raster output first.')
print('SVG:', vector_export.status, vector_export.result == SVG_MARKUP, vector_export.error) if vector_export else print('List SVG output first.')
if raster_export and raster_export.status == 'completed':
    display(raster_export.result.resize((128, 128)))


In [ ]:
data_listing = list_outputs('output-data')


In [ ]:
print(data_listing.status, data_listing.result, data_listing.error)
data_ref = data_listing.result['outputs'][0] if data_listing.result else None


In [ ]:
data_export = export_output(data_ref['cell_id'], data_ref['output_id'], data_ref['revision'], mime='application/vnd.dataresource+json') if data_ref else None
data_saved = export_output(data_ref['cell_id'], data_ref['output_id'], data_ref['revision'], save_to='auto', mime='application/vnd.dataresource+json') if data_ref else None


In [ ]:
import json
print('Data bytes:', data_export.status, isinstance(data_export.result, bytes), json.loads(data_export.result.decode('utf-8')) == DATA_RESOURCE, data_export.media['mime_type'], data_export.error) if data_export and data_export.result else print('Data export pending or unavailable')
print('Saved data:', data_saved.status, data_saved.media.get('path') if data_saved.media else None, data_saved.error) if data_saved else print('No data save')


## A small drawn canvas

The next cell creates a 64×32 canvas and draws red and blue blocks using a short local drawing command. It is an ordinary notebook output, not an app or widget. Some JupyterLab sanitizers remove or disable canvas content; the list/capture tools must report that result honestly. Do not substitute a simulated canvas.


In [ ]:
display(HTML('<canvas id="catalog-output-canvas" width="64" height="32" aria-label="Disposable red and blue canvas"></canvas>'))
display(Javascript("const canvas = document.getElementById('catalog-output-canvas'); if (canvas) { const context = canvas.getContext('2d'); context.fillStyle = '#d4402f'; context.fillRect(0, 0, 32, 32); context.fillStyle = '#2542a3'; context.fillRect(32, 0, 32, 32); }"))


In [ ]:
canvas_listing = list_outputs('output-canvas')

In [ ]:
print(canvas_listing.status, canvas_listing.result, canvas_listing.error)
canvas_output_ref = canvas_listing.result['outputs'][0] if canvas_listing.result and canvas_listing.result['outputs'] else None

In [ ]:
canvases = list_canvases(canvas_output_ref['cell_id'], canvas_output_ref['output_id'], canvas_output_ref['revision']) if canvas_output_ref else None

In [ ]:
print(canvases.status, canvases.result, canvases.error) if canvases else print('List the canvas output first.')
canvas_ref = canvases.result['canvases'][0] if canvases and canvases.result and canvases.result.get('canvases') else None

In [ ]:
canvas_still = capture_canvas(canvas_ref) if canvas_ref else None

In [ ]:
print(canvas_still.status, getattr(canvas_still.result, 'size', None), canvas_still.error) if canvas_still else print('Canvas unavailable in this renderer.')
if canvas_still and canvas_still.status == 'completed':
    print('Canvas pixel (4, 4):', canvas_still.result.getpixel((4, 4)))
    display(canvas_still.result.resize((128, 128)))


In [ ]:
canvas_saved = export_canvas(canvas_ref) if canvas_ref else None

In [ ]:
print(canvas_saved.status, canvas_saved.media, canvas_saved.error) if canvas_saved else print('Canvas unavailable in this renderer.')

In [ ]:
canvas_source = start_canvas(canvas_ref, frame_rate=12) if canvas_ref else None

In [ ]:
print(canvas_source.status, canvas_source.result, canvas_source.error) if canvas_source else print('Canvas unavailable in this renderer.')
print('The cleanup cell below stops an opened source through the shared source control.')

## Capture a supported visible output region

Scroll so the raster output below is completely visible. The region includes its displayed output rectangle only, not code or inputs. Unsupported or partly hidden output is an explicit error.

In [ ]:
display(Image(data=PNG_BYTES))

In [ ]:
region = capture_notebook_region(['region-raster'])

In [ ]:
print(region.status, getattr(region.result, 'size', None), region.error)
if region.status == 'completed':
    display(region.result.resize((128, 128)))


In [ ]:
if canvas_source and canvas_source.result and canvas_source.result.get('source_id'):
    stopped_source = stop_source(canvas_source.result['source_id'])
released_outputs = {}
seen_media_ids = set()
for label, receipt in [
    ('raster export', raster_export), ('vector export', vector_export),
    ('data export', data_export), ('saved data', data_saved),
    ('canvas still', canvas_still), ('canvas export', canvas_saved),
    ('region', region),
]:
    descriptor = receipt.media if receipt and receipt.status == 'completed' else None
    media_id = descriptor.get('media_id') if isinstance(descriptor, dict) else None
    if media_id and media_id not in seen_media_ids:
        seen_media_ids.add(media_id)
        released_outputs[label] = release_media(media_id)
for receipt in (data_saved, canvas_saved):
    descriptor = receipt.media if receipt and receipt.status == 'completed' else None
    if isinstance(descriptor, dict) and descriptor.get('path'):
        saved_path = Path.cwd() / descriptor['path']
        if saved_path.is_file() and hashlib.sha256(saved_path.read_bytes()).hexdigest() == descriptor['sha256']:
            saved_path.unlink()
print('Requested release for', len(released_outputs), 'owned outputs; removed matching generated files.')


In [ ]:
print('Canvas source:', stopped_source.status, stopped_source.error) if 'stopped_source' in globals() else print('No canvas source to stop.')
for label, receipt in released_outputs.items():
    print('Release', label, receipt.status, receipt.error)


## Ask ChatGPT about the live outputs

Run these questions one at a time. Each browser action first returns an operation ID and a running or waiting state; run its later status question after the media row completes before using its output ID, canvas ID, source ID, or managed media ID. Select text for the selection question, and scroll `region-raster` fully into view for region capture.


Use &`read_notebook_view` for this open notebook. Report only its immediate operation ID/state; the later status question will describe the view.


Use &`operation_status` on the preceding `read_notebook_view` operation ID after its media row completes. Report the actual notebook and active-cell facts; if still running, stop and retry later.


Use &`read_selection` with `max_chars=120` for text I selected in this editor. Report the immediate operation ID/state, not selected text yet.


Use &`operation_status` on that `read_selection` operation ID. Quote only the actual bounded selected text, or report an empty selection honestly.


Use &`list_outputs` for `output-text` without rerunning it. Report its immediate operation ID/state; do not invent an output ID before completion.


Use &`operation_status` on that `list_outputs` operation ID after completion. Report the actual output ID, revision, and MIME type; if still running, retry later.


Use &`read_output` on the exact cell ID, output ID, and revision in $`text_ref`, with `max_chars=120`. Report its immediate operation ID/state, not the text yet.


Use &`operation_status` on that `read_output` operation ID after completion. Quote the returned short text and MIME type, or report the actual error.


Use &`export_output` on the exact cell ID, output ID, and revision in $`raster_ref`, with `save_to=None`. Report the immediate operation ID/state; no file should be written.


Use &`operation_status` on that `export_output` operation ID after completion. Report actual image dimensions, MIME type, SHA-256, and managed media ID if provided; do not put pixels in text.


Use &`list_canvases` on the exact cell ID, output ID, and revision in $`canvas_output_ref`. Report its immediate operation ID/state. If this renderer strips the canvas, do not claim one exists.


Use &`operation_status` on the prior `list_canvases` operation ID. Report the actual canvas ID/descriptor only if completed; otherwise describe the real unsupported or pending result.


If the preceding status returned a supported canvas, use &`capture_canvas` on that exact descriptor. Report only its immediate operation ID/state; otherwise stop without making up a canvas.


Use &`operation_status` on the `capture_canvas` operation ID after completion. Report actual dimensions, encoded hash, and managed media ID if returned; the direct Python inspection above shows the sampled pixel. If still running or unsupported, say so.


If a supported canvas descriptor was verified above, use &`export_canvas` on it with `save_to=None`. Report the immediate operation ID/state; otherwise stop without claiming a file or image.


Use &`operation_status` on that `export_canvas` operation ID after completion. Report actual image dimensions, hash, and managed media ID if returned; this in-memory request writes no file.


If a supported canvas descriptor was verified above, use &`start_canvas` at 12 frames per second. Report only its immediate operation ID/state; otherwise stop.


Use &`operation_status` on the `start_canvas` operation ID. Report the actual source ID only if completed; otherwise stop and retry later.


Use &`stop_source` only on the exact canvas source ID verified in the previous status. Report its immediate stop operation ID/state.


With `region-raster` fully visible, use &`capture_notebook_region` for that one cell ID. Report its immediate operation ID/state; do not capture other content.


Use &`operation_status` on the prior region-capture operation ID after completion. Report actual dimensions and managed media ID, or a concrete visibility error, without inventing pixels.


Use &`release_media` only on exact managed media IDs reported by completed AI export, canvas capture/export, and region-capture statuses above. Report each release operation ID/state. Skip missing IDs rather than guessing. Saved notebook displays remain visible.


After the release rows finish, use &`operation_status` on each exact release operation ID from the prior answer. Report actual terminal states; do not claim release from initial snapshots.
